# Setup

In [1]:
# Install (run once at the top of the notebook, in its own cell)
!pip install transformer_lens fancy_einsum einops --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.6 MB/s eta 0:00:00


In [2]:
try:
  import google.colab
  IN_COLAB = True
  print("Running as a Colab notebook")
  %pip install git+https://github.com/neelnanda-io/Easy-Transformer.git@clean-transformer-demo
  # Install another version of node that makes PySvelte work way faster
  !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash -; sudo apt-get install -y nodejs
  %pip install git+https://github.com/neelnanda-io/PySvelte.git
  %pip install fancy_einsum
  %pip install einops
except:
  IN_COLAB = False
  print("Running as a Jupyter notebook - intended for development only!")

Running as a Colab notebook
  Cloning https://github.com/neelnanda-io/Easy-Transformer.git (to revision clean-transformer-demo) to /tmp/pip-req-build-rm85zzcb
  Running command git clone --filter=blob:none --quiet https://github.com/neelnanda-io/Easy-Transformer.git /tmp/pip-req-build-rm85zzcb
  Running command git checkout -b clean-transformer-demo --track origin/clean-transformer-demo
  Switched to a new branch 'clean-transformer-demo'
  Branch 'clean-transformer-demo' set up to track remote branch 'clean-transformer-demo' from 'origin'.
  Resolved https://github.com/neelnanda-io/Easy-Transformer.git to commit 1f25219e631aeb478d17075d47274db32c874e88
  Preparing metadata (setup.py) ... done
  Created wheel for easy_transformer: filename=easy_transformer-0.1.0-py3-none-any.whl size=55601 sha256=e5c25e5375d507231697effe6efd33806bde341e5faf383118340536f44dc35f
  Stored in directory: /tmp/pip-ephem-wheel-cache-mwjyhvyx/wheels/93/f3/71/f103ceb7ff1dea0b7c7d213d85708cfeb9bd35e10f18542b19
Su

In [3]:
!pip install transformer_lens

  Attempting uninstall: typeguard
    Found existing installation: typeguard 2.13.3
    Uninstalling typeguard-2.13.3:
      Successfully uninstalled typeguard-2.13.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pysvelte 1.0.0 requires typeguard~=2.0, but you have typeguard 4.5.1 which is incompatible.


In [4]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
from transformer_lens import HookedTransformer
import torch
import torch.nn as nn
import numpy as np
import math
from transformer_lens.utils import gelu_new, tokenize_and_concatenate, get_corner
import tqdm.auto as tqdm

/tmp/ipykernel_3440/1986855516.py:9: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens.utils import gelu_new, tokenize_and_concatenate, get_corner


In [5]:
reference_gpt2 = HookedTransformer.from_pretrained("gpt2-small", fold_ln=False, center_unembed=False, center_writing_weights=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer


Run a reference forward pass so we have a `cache` for the tests.

In [6]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text).cuda()
logits, cache = reference_gpt2.run_with_cache(tokens)

## Reference activation shapes

Key:
```
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 3072 (4 * d_model)
d_head = 64 (d_model / n_heads)
```

In [7]:
for activation_name, activation in cache.cache_dict.items():
    # Only print for first layer
    if ".0." in activation_name or "blocks" not in activation_name:
        print(activation_name, activation.shape)

hook_embed torch.Size([1, 35, 768])
hook_pos_embed torch.Size([1, 35, 768])
blocks.0.hook_resid_pre torch.Size([1, 35, 768])
blocks.0.ln1.hook_scale torch.Size([1, 35, 1])
blocks.0.ln1.hook_normalized torch.Size([1, 35, 768])
blocks.0.attn.hook_q torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_k torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_v torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_attn_scores torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_pattern torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_z torch.Size([1, 35, 12, 64])
blocks.0.hook_attn_out torch.Size([1, 35, 768])
blocks.0.hook_resid_mid torch.Size([1, 35, 768])
blocks.0.ln2.hook_scale torch.Size([1, 35, 1])
blocks.0.ln2.hook_normalized torch.Size([1, 35, 768])
blocks.0.mlp.hook_pre torch.Size([1, 35, 3072])
blocks.0.mlp.hook_post torch.Size([1, 35, 3072])
blocks.0.hook_mlp_out torch.Size([1, 35, 768])
blocks.0.hook_resid_post torch.Size([1, 35, 768])
ln_final.hook_scale torch.Size([1, 35, 1])
ln_final.hook_normalized torc

## Reference parameter shapes

In [8]:
for name, param in reference_gpt2.named_parameters():
    # Only print for first layer
    if ".0." in name or "blocks" not in name:
        print(name, param.shape)

embed.W_E torch.Size([50257, 768])
pos_embed.W_pos torch.Size([1024, 768])
blocks.0.ln1.w torch.Size([768])
blocks.0.ln1.b torch.Size([768])
blocks.0.ln2.w torch.Size([768])
blocks.0.ln2.b torch.Size([768])
blocks.0.attn.W_Q torch.Size([12, 768, 64])
blocks.0.attn.W_O torch.Size([12, 64, 768])
blocks.0.attn.b_Q torch.Size([12, 64])
blocks.0.attn.b_O torch.Size([768])
blocks.0.attn.W_K torch.Size([12, 768, 64])
blocks.0.attn.W_V torch.Size([12, 768, 64])
blocks.0.attn.b_K torch.Size([12, 64])
blocks.0.attn.b_V torch.Size([12, 64])
blocks.0.mlp.W_in torch.Size([768, 3072])
blocks.0.mlp.b_in torch.Size([3072])
blocks.0.mlp.W_out torch.Size([3072, 768])
blocks.0.mlp.b_out torch.Size([768])
ln_final.w torch.Size([768])
ln_final.b torch.Size([768])
unembed.W_U torch.Size([768, 50257])
unembed.b_U torch.Size([50257])


## Config

In [45]:

@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12

cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


Key:
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 4 * 768 = 3072
d_head = 768 / 12 = 64

In [10]:
# All activation shapes of ref model
for activation_name, activation in cache.cache_dict.items():
  if ".0." in activation_name or "blocks" not in activation_name:
    print(activation_name, activation.shape)

hook_embed torch.Size([1, 35, 768])
hook_pos_embed torch.Size([1, 35, 768])
blocks.0.hook_resid_pre torch.Size([1, 35, 768])
blocks.0.ln1.hook_scale torch.Size([1, 35, 1])
blocks.0.ln1.hook_normalized torch.Size([1, 35, 768])
blocks.0.attn.hook_q torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_k torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_v torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_attn_scores torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_pattern torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_z torch.Size([1, 35, 12, 64])
blocks.0.hook_attn_out torch.Size([1, 35, 768])
blocks.0.hook_resid_mid torch.Size([1, 35, 768])
blocks.0.ln2.hook_scale torch.Size([1, 35, 1])
blocks.0.ln2.hook_normalized torch.Size([1, 35, 768])
blocks.0.mlp.hook_pre torch.Size([1, 35, 3072])
blocks.0.mlp.hook_post torch.Size([1, 35, 3072])
blocks.0.hook_mlp_out torch.Size([1, 35, 768])
blocks.0.hook_resid_post torch.Size([1, 35, 768])
ln_final.hook_scale torch.Size([1, 35, 1])
ln_final.hook_normalized torc

# Actual Implementation

In [11]:
for name, param in reference_gpt2.named_parameters():
  print(name, param.shape)


embed.W_E torch.Size([50257, 768])
pos_embed.W_pos torch.Size([1024, 768])
blocks.0.ln1.w torch.Size([768])
blocks.0.ln1.b torch.Size([768])
blocks.0.ln2.w torch.Size([768])
blocks.0.ln2.b torch.Size([768])
blocks.0.attn.W_Q torch.Size([12, 768, 64])
blocks.0.attn.W_O torch.Size([12, 64, 768])
blocks.0.attn.b_Q torch.Size([12, 64])
blocks.0.attn.b_O torch.Size([768])
blocks.0.attn.W_K torch.Size([12, 768, 64])
blocks.0.attn.W_V torch.Size([12, 768, 64])
blocks.0.attn.b_K torch.Size([12, 64])
blocks.0.attn.b_V torch.Size([12, 64])
blocks.0.mlp.W_in torch.Size([768, 3072])
blocks.0.mlp.b_in torch.Size([3072])
blocks.0.mlp.W_out torch.Size([3072, 768])
blocks.0.mlp.b_out torch.Size([768])
blocks.1.ln1.w torch.Size([768])
blocks.1.ln1.b torch.Size([768])
blocks.1.ln2.w torch.Size([768])
blocks.1.ln2.b torch.Size([768])
blocks.1.attn.W_Q torch.Size([12, 768, 64])
blocks.1.attn.W_O torch.Size([12, 64, 768])
blocks.1.attn.b_Q torch.Size([12, 64])
blocks.1.attn.b_O torch.Size([768])
blocks.1.a

## Some tests

In [27]:

# for our model (where input is floats)
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randn(shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

# for our model (where input is ints)
def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randint(100, 1000, shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

# takes instance of that layer from ref model and its original input
def load_gpt2_test(cls, gpt2_layer, input_name, cache_dict=cache.cache_dict):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    # Allow inputs of strings or tensors
    if isinstance(input_name, str):
        reference_input = cache_dict[input_name]
    else:
        reference_input = input_name
    print("Input shape:", reference_input.shape)
    output = layer(reference_input)
    print("Output shape:", output.shape)

    # Attention has different set of inputs compared to other layers
    if cls.__name__ == "Attention":
      reference_output = gpt2_layer(reference_input, reference_input, reference_input)
    else:
      reference_output = gpt2_layer(reference_input)

    print("Reference output shape: ", reference_output.shape)

    comparison = torch.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of the values are correct")
    return output

## LayerNorm

1.  Make mean 0
2. normalize to have variance 1
3. Scale with learned weights
4. Translate with learned bias

In [28]:
class LayerNorm(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.w = nn.Parameter(torch.ones(cfg.d_model))
    self.b = nn.Parameter(torch.zeros(cfg.d_model))

  def forward(self, residual):
    # residual: [batch, position, d_model]
    if cfg.debug: print("Residual:", residual.shape)
    residual = residual - einops.reduce(residual, "batch position d_model -> batch position 1", "mean") # making mean 0
    # Calculate variance, then sqrt it. Epsilon to prevent divide by 0
    scale = (einops.reduce(residual.pow(2), "batch position d_model -> batch position 1", "mean") + cfg.layer_norm_eps).sqrt()
    normalized = residual / scale
    normalized = normalized * self.w + self.b
    if cfg.debug: print("Normalized:", residual.shape)
    return normalized


In [29]:
# Testing layernorm
_ = rand_float_test(LayerNorm, [2, 4, 768])

Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])



In [30]:
_ = load_gpt2_test(LayerNorm, reference_gpt2.ln_final, "blocks.11.hook_resid_post")

Input shape: torch.Size([1, 35, 768])
Residual: torch.Size([1, 35, 768])
Normalized: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


## Embedding
A lookup table from tokens to residual stream vectors

In [20]:
class Embed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_E = nn.Parameter(torch.empty((cfg.d_vocab, cfg.d_model)))
    nn.init.normal_(self.W_E, std = self.cfg.init_range)

  def forward(self, tokens):
    # tokens shape: [batch, positions]
    if cfg.debug: print("Tokens", tokens.shape)
    embed = self.W_E[tokens, :] # applying the embedding by indexing along d vocab axis, final shape = [batch, pos, d_model]
    if cfg.debug: print("Embeddings", embed.shape)
    return embed

In [31]:
# Testing embedding layer
rand_int_test(Embed, [2, 4])
load_gpt2_test(Embed, reference_gpt2.embed, tokens)

Input shape: torch.Size([2, 4])
Tokens torch.Size([2, 4])
Embeddings torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35])
Tokens torch.Size([1, 35])
Embeddings torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207],
         [ 0.1474, -0.0959,  0.1430,  ...,  0.1030, -0.0625, -0.1131],
         [ 0.1596, -0.1249,  0.1148,  ...,  0.2558,  0.0196,  0.0145],
         ...,
         [-0.0393,  0.0050,  0.0421,  ..., -0.0477,  0.0670, -0.0471],
         [-0.1488,  0.1519,  0.0056,  ..., -0.3107,  0.2073,  0.0377],
         [-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453]]],
       device='cuda:0', grad_fn=<IndexBackward0>)

## Positional Embedding

lookup table for positions to give each position a context instead of just each word/token like bag of words

this weight is updated on gradient descent hence the name learned absolute pos embedding

In [32]:
class PosEmbed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_pos = nn.Parameter(torch.empty((cfg.n_ctx, cfg.d_model)))
    nn.init.normal_(self.W_pos, std=self.cfg.init_range)

  def forward(self, tokens):
    # tokens is [batch, position]
    if cfg.debug: print("Tokens:", tokens.shape)
    pos_embed = self.W_pos[:tokens.size(1), :] # [position, d_model] -> indexing by position, so taking the size of the seq
    pos_embed = einops.repeat(pos_embed, "position d_model -> batch position d_model", batch = tokens.size(0))
    if cfg.debug: print("pos_embed:", pos_embed.shape)
    return pos_embed


In [33]:
# Testing pos embed layer
rand_int_test(PosEmbed, [2, 4])
load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)

Input shape: torch.Size([2, 4])
Tokens: torch.Size([2, 4])
pos_embed: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35])
Tokens: torch.Size([1, 35])
pos_embed: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[-1.8821e-02, -1.9742e-01,  4.0267e-03,  ..., -4.3044e-02,
           2.8267e-02,  5.4490e-02],
         [ 2.3959e-02, -5.3792e-02, -9.4879e-02,  ...,  3.4170e-02,
           1.0172e-02, -1.5573e-04],
         [ 4.2161e-03, -8.4764e-02,  5.4515e-02,  ...,  1.9745e-02,
           1.9325e-02, -2.1424e-02],
         ...,
         [ 4.6277e-04,  2.3037e-02,  4.1227e-02,  ..., -1.9287e-03,
          -2.3037e-03, -4.3189e-03],
         [-2.7136e-03,  2.1724e-02,  3.9675e-02,  ...,  4.2048e-04,
          -4.8160e-03, -9.2252e-04],
         [ 6.6815e-03,  2.0595e-02,  3.6596e-02,  ..., -9.5090e-04,
          -3.2512e-03, -9.6509e-04]]], device='cuda:0',
       grad_fn=<ExpandBackward0>)

## Attention
1. Produce an attention pattern for each destination token - a probability dist over 0th to curr token
  * Linear map from input -> query, key, where shape: [batch, head_index, d_head]
  * then dot product every pair of queries and keys to get attention scores [batch, head_index, query_pos, key_pos] (query = dest, key = src)
  * Scale and mask attn scores to make it causal
  * softmax row-wise, to get a probability dist along each the key_pos dim -> this is the final attention pattern
2. Move info from src tokens to dest token using attention pattern (moving is via linear map)
-  Linear map from input -> value [batch, key_pos, head_index, d_head]
- Mix along the key_pos axis with attention pattern to get a mixed value [batch, query_pos, head_index, d_head]
- map to output, [batch, position, d_model] (position is query pos since we summed over all the attention heads


In [34]:
class Attention(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_Q = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_Q, std=self.cfg.init_range)
    self.b_Q = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
    self.W_K = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_K, std=self.cfg.init_range)
    self.b_K = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
    self.W_V = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_V, std=self.cfg.init_range)
    self.b_V = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))

    self.W_O = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
    nn.init.normal_(self.W_O, std=self.cfg.init_range)
    self.b_O = nn.Parameter(torch.zeros((cfg.d_model)))

    self.register_buffer("IGNORE", torch.tensor(-1e5, dtype=torch.float32, device="cuda"))
  def forward(self, normalized_resid_pre):
    # normalized_resid_pre: [batch, position, d_model]
    if self.cfg.debug: print("Normalized_resid_pre:", normalized_resid_pre.shape)

    q = einsum("batch query_pos d_model, n_heads d_model d_head -> batch query_pos n_heads d_head", normalized_resid_pre, self.W_Q) + self.b_Q
    k = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_K) + self.b_K

    attn_scores = einsum("batch query_pos n_heads d_head, batch key_pos n_heads d_head -> batch n_heads query_pos key_pos", q, k)
    attn_scores = attn_scores / math.sqrt(self.cfg.d_head)
    attn_scores = self.apply_causal_mask(attn_scores)

    pattern = attn_scores.softmax(dim=-1) # [batch, n_head, query_pos, key_pos]

    v = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_V) + self.b_V

    z = einsum("batch n_heads query_pos key_pos, batch key_pos n_heads d_head -> batch query_pos n_heads d_head", pattern, v)

    attn_out = einsum("batch query_pos n_heads d_head, n_heads d_head d_model -> batch query_pos d_model", z, self.W_O) + self.b_O

    if cfg.debug:
      print("z shape:", z.shape)
      print("W_O shape:", self.W_O.shape)

    return attn_out

  # mask to remaining tokens in a sequence to maintain backward looking property properly
  def apply_causal_mask(self, attn_scores):
    mask = torch.triu(torch.ones(attn_scores.size(-2), attn_scores.size(-1), device=attn_scores.device), diagonal = 1).bool()
    attn_scores.masked_fill_(mask, self.IGNORE)
    return attn_scores



In [35]:
# Testing attention layer
rand_float_test(Attention, [2, 4, 768])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["blocks.0.ln1.hook_normalized"])

Input shape: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35, 768])
Normalized_resid_pre: torch.Size([1, 35, 768])
z shape: torch.Size([1, 35, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[ 1.3649e+00,  2.1711e+00,  7.0824e+00,  ..., -1.4679e-01,
           2.6480e-01,  9.8746e-01],
         [-1.3159e+01, -4.1196e+00,  8.6870e+00,  ..., -4.7698e-01,
          -2.4685e-01,  3.7986e-01],
         [-1.7002e+01,  4.8321e+00, -6.2118e-01,  ..., -7.1945e-01,
           1.0781e+00,  5.4464e-01],
         ...,
         [-1.3211e+01,  7.5173e-01,  8.9662e+00,  ..., -4.2861e-01,
           4.6559e-01, -9.4983e-01],
         [-1.3985e-03,  6.5740e+00,  1.9785e+01,  ..., -6.7092e-01,
          -1.0935e-01,  7.8003e-02],
         [-6.0138e+00, -1.8512e-01,  1.8866e+01,  ..., -5.4550e-01,
          -4.9668e-02, -1.4721e-01]]], device='cuda:0', grad_fn=<AddBackward0>)

## MLP

In [50]:
class MLP(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_in = nn.Parameter(torch.empty((cfg.d_model, cfg.d_mlp)))
    nn.init.normal_(self.W_in, std=self.cfg.init_range)
    self.b_in = nn.Parameter(torch.zeros(cfg.d_mlp))

    self.W_out = nn.Parameter(torch.empty((cfg.d_mlp, cfg.d_model)))
    nn.init.normal_(self.W_out, std=self.cfg.init_range)
    self.b_out = nn.Parameter(torch.zeros(cfg.d_model))

  def forward(self, normalized_resid_mid):
    # normalized resid mid stream: [batch, position, d_model]
    if cfg.debug: print("Normalized residual stream mid:", normalized_resid_mid.shape)

    pre = einsum("batch position d_model, d_model d_mlp -> batch position d_mlp", normalized_resid_mid, self.W_in) + self.b_in
    post = gelu_new(pre)
    mlp_out = einsum("batch position d_mlp, d_mlp d_model -> batch position d_model", post, self.W_out) + self.b_out
    return mlp_out






In [39]:
# Testing the mlp layer

rand_float_test(MLP, [2, 4, 768])
load_gpt2_test(MLP, reference_gpt2.blocks[0].mlp, cache["blocks.0.ln1.hook_normalized"])

Input shape: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35, 768])
Normalized residual stream mid: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[  0.0274,   2.4044,   2.6138,  ...,  13.3645,   8.4272,  -0.8360],
         [ -8.6545,   1.7920,   0.6268,  ...,  -5.1794,  -1.2061,   5.3991],
         [-10.3536, -13.2765,  -6.0768,  ...,   4.8616,  -1.8622,  10.4763],
         ...,
         [ -8.7080,   7.6277,   6.4608,  ...,   4.3486,  -5.2200,   9.6780],
         [-11.1025,   2.4969,  21.4818,  ...,  -0.5803,  -2.8098,  12.2156],
         [  3.2533,  -0.9791,  16.5849,  ...,   6.3753,   8.0988,   8.1493]]],
       device='cuda:0', grad_fn=<AddBackward0>)

## Transformer Block

In [54]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg

    self.ln1 = LayerNorm(cfg)
    self.attn = Attention(cfg)
    self.ln2 = LayerNorm(cfg)
    self.mlp = MLP(cfg)

  def forward(self, resid_pre):
    # Resid pre is shape [batch, position, d_model]

    # Attention layer and updating residual stream
    normalized_resid_pre = self.ln1(resid_pre)
    attn_out = self.attn(normalized_resid_pre)
    resid_mid = resid_pre + attn_out

    # MLP layer and updating final resid stream output
    normalized_resid_mid = self.ln2(resid_mid)
    mlp_out = self.mlp(normalized_resid_mid)
    resid_out = resid_mid + mlp_out

    return resid_out

In [55]:
# Testing one transformer block

rand_float_test(TransformerBlock, [2, 4, 768])
load_gpt2_test(TransformerBlock, reference_gpt2.blocks[0], cache["resid_pre", 0])

Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35, 768])
Residual: torch.Size([1, 35, 768])
Normalized: torch.Size([1, 35, 768])
Normalized_resid_pre: torch.Size([1, 35, 768])
z shape: torch.Size([1, 35, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([1, 35, 768])
Normalized: torch.Size([1, 35, 768])
Normalized residual stream mid: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[ 0.3911,  0.1543,  0.6005,  ...,  1.7198,  1.7365,  0.3930],
         [-0.9039, -0.0360,  0.2351,  ..., -0.4148,  0.3562,  0.3936],
         [-0.9647, -2.4819, -1.4995,  ...,  1.4046,  0.7616,  0.5918],
         ...,
         [-0.7421,  0.9251, -0.3218,  ...,  0.2921,  0.1097, -0.5344],
         [-1.3221,  0.8960,  1.1795,  ..., -0.5544, -0.4071,  0.9255],
         [ 1.1209, -0.8919,  1.3737,  ..., -0.1356,  0.3434,  0.4517]]],
       device='cuda:0', grad_fn=<AddBackward0>)

## Unembedding

In [56]:
class Unembed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_U = nn.Parameter(torch.empty((cfg.d_model, cfg.d_vocab)))
    nn.init.normal_(self.W_U, std=self.cfg.init_range)
    self.b_U = nn.Parameter(torch.zeros((cfg.d_vocab)))

  def forward(self, normalized_resid_final):
    # normalized resid final/out = [batch, position, d_model]
    if cfg.debug: print("Normalized_resid_final:", normalized_resid_final.shape)
    logits = einsum("batch position d_model, d_model d_vocab -> batch position d_vocab", normalized_resid_final, self.W_U) + self.b_U
    return logits


## Full Transformer

In [65]:
class DemoTransformer(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.embed = Embed(cfg)
    self.pos_embed = PosEmbed(cfg)
    self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
    self.ln_final = LayerNorm(cfg)
    self.unembed = Unembed(cfg)

  '''
  def forward(self, tokens):
    # tokens [batch, position]
    embed = self.embed(tokens)
    pos_embed = self.pos_embed(tokens)
    resid = embed + pos_embed
    for block in self.blocks:
      resid = block(resid)
    normalized_resid_final = self.ln_final(resid)
    logits = self.unembed(normalized_resid_final)
    # final shape: [batch, position, logits]
    return logits
'''
  # forwarding logic but saving mid outputs to a cache
  def forward(self, tokens, return_cache=False):
    # tokens [batch, position]
    cache = {}
    embed = self.embed(tokens)
    pos_embed = self.pos_embed(tokens)
    resid = embed + pos_embed
    cache["resid_initial"] = resid

    for i, block in enumerate(self.blocks):
        normalized_resid_pre = block.ln1(resid)
        attn_out = block.attn(normalized_resid_pre)
        resid_mid = resid + attn_out
        cache[f"block_{i}.resid_mid"] = resid_mid

        normalized_resid_mid = block.ln2(resid_mid)
        mlp_out = block.mlp(normalized_resid_mid)
        resid = resid_mid + mlp_out
        cache[f"block_{i}.resid_post"] = resid

  # final shape: [batch, position, logits]
    normalized_resid_final = self.ln_final(resid)
    logits = self.unembed(normalized_resid_final)
    return (logits, cache) if return_cache else logits


In [66]:
rand_int_test(DemoTransformer, [2, 4])
load_gpt2_test(DemoTransformer, reference_gpt2, tokens)

Input shape: torch.Size([2, 4])
Tokens torch.Size([2, 4])
Embeddings torch.Size([2, 4, 768])
Tokens: torch.Size([2, 4])
pos_embed: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4

tensor([[[ -43.4317,  -39.8364,  -43.0660,  ...,  -54.0877,  -54.3452,
           -42.3644],
         [-128.0392, -127.9936, -130.7010,  ..., -136.7122, -129.9261,
          -129.3966],
         [-119.8520, -121.0063, -123.8820,  ..., -128.5180, -126.6027,
          -121.9060],
         ...,
         [-112.9815, -112.7748, -117.0634,  ..., -121.2914, -117.6574,
          -114.5005],
         [ -98.6724, -104.4888, -108.7361,  ..., -118.3552, -113.8766,
          -106.3604],
         [-126.8285, -128.9596, -128.3941,  ..., -140.1970, -138.5883,
          -122.3697]]], device='cuda:0', grad_fn=<AddBackward0>)

# Trying out the model

In [73]:
demo_gpt2 = DemoTransformer(Config(debug=False))
demo_gpt2.cfg.debug = False
demo_gpt2.load_state_dict(reference_gpt2.state_dict(), strict=False)
demo_gpt2.cuda()

DemoTransformer(
  (embed): Embed()
  (pos_embed): PosEmbed()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNorm()
      (attn): Attention()
      (ln2): LayerNorm()
      (mlp): MLP()
    )
  )
  (ln_final): LayerNorm()
  (unembed): Unembed()
)

In [74]:
test_thing = """In just a few short years, rapper Playboi Carti has amassed over 19B streams worldwide. Carti has been unstoppable since the release of his 2017 single "Magnolia," as its meteoric rise garnered cosigns from Beyoncé and features on series like Atlanta. His self-titled album has accumulated nearly 7.8B streams to date after debuting at #12 on the Billboard 200 chart where it spent 63 weeks. The following year, Carti dropped his album Die Lit, debuting at #3 on the Billboard 200, with collaborations like Lil Uzi Vert on "Shoota,"
"Poke It Out" with Nicki Minaj & "Love Hurts" with Travis Scott. The album has nearly 9.5B global streams to date and spent a total of 11 weeks on the Billboard 200. In April of 2020, Playboi Carti returned with track "@MEH," and on Christmas day, he landed his first #1 album on Billboard's 200 Chart with Whole Lotta Red which has amassed a staggering 9.4B global streams to date. Playboi Carti kicked off 2024 strong with new music & collaborations including
"CARNIVAL" with Kanye West, Ty Dolla $ign, and Rich the Kid, reaching #1 on the Billboard Hot 100 chart. Carti's collaboration with Travis Scott on their song "FE!N" peaked at #5 on Billboard's Hot 100. Playboi Carti has also recently featured on "I LUV IT" with Camila Cabello, "TYPE SHIT" with Future, Metro Boomin, and Travis Scott, and "Popular" with The Weeknd and Madonna. His most recent collaboration "Timeless" with The Weeknd debuted at #3 ol Billboard Hot 100."""

In [75]:
tokens_demo = reference_gpt2.to_tokens(test_thing).cuda()
demo_logits = demo_gpt2(tokens_demo)

Tokens torch.Size([1, 367])
Embeddings torch.Size([1, 367, 768])
Tokens: torch.Size([1, 367])
pos_embed: torch.Size([1, 367, 768])
Residual: torch.Size([1, 367, 768])
Normalized: torch.Size([1, 367, 768])
z shape: torch.Size([1, 367, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([1, 367, 768])
Normalized: torch.Size([1, 367, 768])
Normalized residual stream mid: torch.Size([1, 367, 768])
Residual: torch.Size([1, 367, 768])
Normalized: torch.Size([1, 367, 768])
z shape: torch.Size([1, 367, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([1, 367, 768])
Normalized: torch.Size([1, 367, 768])
Normalized residual stream mid: torch.Size([1, 367, 768])
Residual: torch.Size([1, 367, 768])
Normalized: torch.Size([1, 367, 768])
z shape: torch.Size([1, 367, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([1, 367, 768])
Normalized: torch.Size([1, 367, 768])
Normalized residual stream mid: torch.Size([1, 367, 768])
Residual: torch.Size([1, 3

In [78]:
# seeing top prediction for every position

import torch.nn.functional as F

for i in range(cfg.n_layers):
    resid = cache[f"block_{i}.resid_post"]
    normalized = demo_gpt2.ln_final(resid)
    block_logits = demo_gpt2.unembed(normalized)  # [1, 367, d_vocab]

    # Top token at every position
    top_tokens = block_logits[0].argmax(dim=-1)  # [367]
    decoded = reference_gpt2.to_string(top_tokens)

    print(f"=== Block {i} ===")
    print(decoded)
    print()

Residual: torch.Size([1, 367, 768])
Normalized: torch.Size([1, 367, 768])
Normalized_resid_final: torch.Size([1, 367, 768])
=== Block 0 ===

 addition one few hundred short ago which rapper Playboye Cartye been amasseddrivethBS streams worldwide
 Cartye been seen unstoppable since same release the own 2017 singlenoificent Trees while well own meteoric rise garnered cosignalsa the Beyoncé then features top series ours Atlanta And own selfbasedteitled album been accumulated nearlyth But88rick streams be date the debut the least #31 the same Billboard times chartabouts's spentrd ago
 most follows ago which Cartwa dropped own album Die Lit which debuting least #rd the same Billboard times which the collaborations a LilUzi Vert thenoShoot few while
Auppetokeself OutA the Nickwa Minaj/no Love Hurts/" the Travis Scott
 latter album been nearly09 Thetherk warming streams be date then spent few number theth ago the same Billboard times
 addition 2015 the 2020 although Playbopec Cartpec return t